# DenseFlow Pipeline

This notebook demonstrates the full **DenseFlow** pipeline for Ethereum money-laundering detection.

## Pipeline overview

```
Raw transactions (all-normal-tx.csv)
        │
        ▼
Step 1: Preprocessing
        Build address ↔ index maps, compute temporal suspiciousness scores,
        produce the HoloScope input tensor (all-normal-tx_holo.csv)
        │
        ▼
Step 2: HoloScope
        Detect suspicious dense sub-graphs (bipartite, weighted, temporal)
        → candidate suspicious account set  (myheist_k_<K>.xlsx)
        │
        ▼
Step 3: Maximum-flow expansion
        For each HoloScope candidate, trace money flow through the
        transaction network using a max-flow solver
        → expanded suspicious set M  (Result/<case>_out/...xlsx)
        │
        ▼
Step 4: Evaluation
        Measure  Precision · MCR (money-coverage rate) · |M|
```

**Cases included:** `AlphaHomora`, `CryptopiaHacker`, `PlusTokenPonzi`

## 0. Environment setup

In [ ]:
from pathlib import Path
import os, sys

# Ensure the notebook always runs from the Denseflow-clean root
repo_root = Path.cwd()
while repo_root.name not in ('Denseflow-clean', 'Denseflow') and repo_root.parent != repo_root:
    repo_root = repo_root.parent
# If we end up at the repo root that contains Denseflow-clean, step in
if (repo_root / 'Denseflow-clean').exists():
    repo_root = repo_root / 'Denseflow-clean'
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Working directory:', Path.cwd())

In [ ]:
import importlib
import numpy as np
import pandas as pd
import time

# Compatibility shims for older spartan2 code
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'bool'):
    np.bool = bool
if not hasattr(time, 'clock'):
    time.clock = time.perf_counter

import Code.myHoloscope as myHoloscope
import Code.myMaxflow as myMaxflow
import Code.Check as check
import Code.info as info
import Code.holodatatran as holo
import Code.Datatran_1 as dt

importlib.reload(myHoloscope)
importlib.reload(myMaxflow)

from Code.myHoloscope import *
from Code.myMaxflow import *
from Code.Check import *
from Code.info import *

print('Imports OK.  Available cases:', CASES)

---
## 1. Preprocessing

For each case we need:
1. `inputData/AML/<case>/all-normal-tx.csv`  – raw transactions  *(already present)*
2. `inputData/AML/data/<case>/all-normal-tx_from.txt` / `_to.txt`  – address ↔ index maps
3. `inputData/AML/<case>/all-normal-tx_holo.csv`  – HoloScope-ready tensor

In [ ]:
import ast

base      = Path('./inputData/AML')
data_base = base / 'data'

# ── helper: ensure address ↔ index maps exist ──────────────────────────────
def ensure_addr_maps(case, normal_tx):
    case_data_dir = data_base / case
    case_data_dir.mkdir(parents=True, exist_ok=True)
    from_file = case_data_dir / 'all-normal-tx_from.txt'
    to_file   = case_data_dir / 'all-normal-tx_to.txt'

    if from_file.exists() and to_file.exists():
        from_list = ast.literal_eval(from_file.read_text(encoding='utf-8'))
        to_list   = ast.literal_eval(to_file.read_text(encoding='utf-8'))
        from_map  = {addr: idx for idx, addr in from_list}
        to_map    = {addr: idx for idx, addr in to_list}
        return from_map, to_map

    df = pd.read_csv(normal_tx)
    df['value'] = pd.to_numeric(df['value'], errors='coerce').fillna(0.0)
    df = df[df['value'] != 0.0].copy()
    df['from'] = df['from'].astype(str)
    df['to']   = df['to'].astype(str)

    from_addrs = pd.unique(df['from'])
    to_addrs   = pd.unique(df['to'])
    from_list  = list(enumerate(from_addrs.tolist()))
    to_list    = list(enumerate(to_addrs.tolist()))
    from_file.write_text(str(from_list), encoding='utf-8')
    to_file.write_text(str(to_list),     encoding='utf-8')
    from_map = {addr: idx for idx, addr in from_list}
    to_map   = {addr: idx for idx, addr in to_list}
    return from_map, to_map


# ── helper: ensure HoloScope tensor CSV exists ─────────────────────────────
def ensure_holo_tx(case, normal_tx, from_map, to_map):
    case_dir  = base / case
    holo_file = case_dir / 'all-normal-tx_holo.csv'
    if holo_file.exists():
        return

    print(f'[{case}] building all-normal-tx_holo.csv …')
    df = pd.read_csv(normal_tx)
    df['value']     = pd.to_numeric(df['value'],     errors='coerce').fillna(0.0)
    df['timeStamp'] = pd.to_numeric(df['timeStamp'], errors='coerce').fillna(0).astype(int)
    df['from']      = df['from'].astype(str)
    df['to']        = df['to'].astype(str)
    df = df[(df['value'] != 0) & df['from'].isin(from_map) & df['to'].isin(to_map)]

    dates = pd.to_datetime(df['timeStamp'], unit='s', utc=True, errors='coerce').dt.strftime('%Y-%m-%d')
    holo_df = pd.DataFrame({
        0: df['from'].map(from_map).astype('int64'),
        1: df['to'].map(to_map).astype('int64'),
        2: dates,
        3: df['value'].astype(float),
        4: 1,
    }).dropna(subset=[2])
    holo_df.to_csv(holo_file)


# ── run preprocessing for every case ───────────────────────────────────────
for casename in CASES:
    print(f'Preparing {casename} …')
    normal_tx = base / casename / 'all-normal-tx.csv'
    from_a2n, to_a2n = ensure_addr_maps(casename, normal_tx)
    ensure_holo_tx(casename, normal_tx, from_a2n, to_a2n)
    print(f'  [{casename}] preparation complete')

---
## 2. HoloScope – dense sub-graph detection

HoloScope identifies suspicious dense sub-graphs in the bipartite, weighted, temporal transaction graph.  
It returns top-k suspicious candidate accounts that serve as seed nodes for the MaxFlow step.

`level` selects which combination of suspiciousness signals is used (topology / time / amount / rating).  
We use `level=1` (topology + timestamps) as the default configuration.

In [ ]:
# Configuration: which k to use per case
caseing   = CASES
caseandk  = {case: 10 for case in CASES}      # default k=10 for all cases
caseandk['PlusTokenPonzi'] = 10

# HoloScope: build graph and run
hs_models = {}
for casename in caseing:
    k = caseandk[casename]
    # Check whether output already exists for this k
    out_xlsx = base / casename / f'myheist_k_{k}.xlsx'
    if out_xlsx.exists():
        print(f'[{casename}] HoloScope results (k={k}) already exist, skipping.')
        continue
    print(f'[{casename}] Building HoloScope model (numSing=10) …')
    hs = build_hs(casename, numSing=10)
    hs_models[casename] = hs
    print(f'[{casename}] Running HoloScope (k={k}) …')
    output_holo(casename, k, hs)
    print(f'[{casename}] HoloScope done → {out_xlsx}')

---
## 3. Maximum-flow expansion

For each HoloScope candidate (per level), we run a max-flow solver from the known source address through the full transaction network.  
All addresses on non-zero flow paths are added to the suspicious set **M**.

The combined set (HoloScope candidates ∪ flow-reachable nodes) is stored in `Result/<case>_out/<case>_k_<K>_level_<L>.xlsx`.

In [ ]:
for casename in caseing:
    k          = caseandk[casename]
    sourceadds = saddset[casename]

    # Build / read the transaction graph
    gpath   = Path('./Maxflow_graph') / casename
    already = (gpath / 'start_nodes.npy').exists()
    if already:
        print(f'[{casename}] Reading pre-built graph …')
        start_nodes, end_nodes, capacities, nodenum = read_g(casename)
    else:
        print(f'[{casename}] Building graph …')
        start_nodes, end_nodes, capacities, nodenum = build_g(casename)

    source = nodenum[sourceadds[0]]

    # Run max-flow for levels 0–3
    for level in range(4):
        out_file = Path('./Result') / f'{casename}_out' / f'{casename}_k_{k}_level_{level}.xlsx'
        if out_file.exists():
            print(f'[{casename}] level={level} result exists, skipping.')
            continue
        print(f'[{casename}] MaxFlow  k={k}  level={level} …')
        output_flow_add(casename, k, level, source)

---
## 4. Evaluation

We evaluate the final suspicious set **M** with three metrics:

| Metric | Definition |
|--------|------------|
| **Precision** | `|detected ∩ true heist| / |detected|` |
| **MCR** (money-coverage rate) | `min(traced ETH / true-illicit ETH, 1)` |
| **\|M\|** | size of the detected suspicious account set |

In [ ]:
results = {}

for casename in caseing:
    k     = caseandk[casename]
    level = 1          # level=1: topology + timestamp signals
    print(f'\n{'='*60}')
    print(f'Case: {casename}  (k={k}, level={level})')
    print('='*60)
    m = check_metrics_my(casename, level=level, k=k)
    if m:
        results[casename] = m

### Summary table

In [ ]:
rows = []
for case, metrics in results.items():
    for method, (pre, mcr, m_size) in metrics.items():
        rows.append({'Case': case, 'Method': method,
                     'Precision': round(pre, 4),
                     'MCR':       round(mcr, 4),
                     '|M|':       m_size})

summary = pd.DataFrame(rows).set_index(['Case', 'Method'])
print(summary.to_string())
summary

---
## 5. Multi-level comparison (optional)

HoloScope supports 7 signal combinations (`level` 0–6).  
Here we sweep all available levels to show how the chosen signals affect precision and MCR.

In [ ]:
level_results = []

for casename in caseing:
    k = caseandk[casename]
    for level in range(4):
        out_file = Path('./Result') / f'{casename}_out' / f'{casename}_k_{k}_level_{level}.xlsx'
        if not out_file.exists():
            continue
        m = check_metrics_my(casename, level=level, k=k)
        if m:
            pre, mcr, m_size = list(m.values())[0]
            level_results.append({'Case': casename, 'Level': level, 'k': k,
                                   'Precision': pre, 'MCR': mcr, '|M|': m_size})

level_df = pd.DataFrame(level_results)
if not level_df.empty:
    print(level_df.to_string(index=False))
    display(level_df)